[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/15.%20The%20Vessel%20Re-stow%20Problem%20%28Transshipment%29/P15-Tier-9_executed.ipynb)

---


# 15. The Vessel Re-stow Problem (Transshipment)
## Tier 9: QAOA as a Quantum Optimization Demonstration

### Purpose and scope
This notebook formulates a small vessel re-stow assignment problem as a Quadratic Unconstrained Binary Optimization (QUBO) model, then uses a state-vector simulation of the Quantum Approximate Optimization Algorithm (QAOA) to sample candidate assignments. The example makes the QUBO-to-circuit connection explicit; it is not evidence of quantum speedup or a model of operational-scale quantum hardware.

The input is a set of containers, their current locations and destinations, and a set of available positions. The output is the lowest-energy feasible assignment observed in sampled QAOA measurements, together with its QUBO energy, move cost, and measurement frequency. Because QAOA is stochastic, repeated runs can produce different samples and results.

### Model assumptions
- Each container must be assigned to exactly one available position, and each position can hold at most one container.
- The objective rewards keeping containers at their current positions, favors placing containers for the same destination in the same bay, and applies a simple preference for placing heavier containers in designated stronger bays.
- The model does not include container accessibility, stacking constraints, stability, crane scheduling, discharge-sequence feasibility, or operational time and labor costs. The `accessible`, `max_weight`, and `occupied` fields are descriptive data fields and are not constraints in this QUBO.
- The move-cost report uses a fixed charge of $150 per container moved. The QUBO energy is a separate weighted objective and is not measured in dollars.

### Method overview
For each container-position pair, a binary variable records whether that assignment is selected. The QUBO combines assignment penalties and the weighted objective in one quadratic energy function. QAOA prepares a uniform superposition, applies alternating problem and mixing operations, and samples bit strings. A classical stochastic parameter search adjusts the QAOA angles using measured energies. Sampled bit strings are decoded, and only assignments satisfying the model's assignment and capacity constraints are eligible as reported solutions.

This notebook uses a state-vector simulator. With `n` binary variables, a state vector has `2^n` complex amplitudes, so memory and computation grow exponentially. The example uses 16 binary variables and therefore requires 16 qubits in this encoding. This simulation is intended for teaching and small instances; it does not establish a quantum advantage, approximation guarantee, or practical hardware requirement.

### QUBO model
Let $i \in \{1,\ldots,N\}$ index containers and $j \in \{1,\ldots,M\}$ index positions. The binary decision variable $x_{ij}$ equals 1 when container $i$ is assigned to position $j$, and 0 otherwise. The assignment and capacity requirements are

$$\sum_{j=1}^{M} x_{ij}=1 \quad (i=1,\ldots,N), \qquad \sum_{i=1}^{N}x_{ij}\leq 1 \quad (j=1,\ldots,M).$$

The first requirement is represented by the squared penalty $A(1-\sum_j x_{ij})^2$. Since $x_{ij}^2=x_{ij}$ for a binary variable, and since the constant $A$ does not change which assignment minimizes the energy, its QUBO contribution is $-A\sum_j x_{ij}+2A\sum_{j<k}x_{ij}x_{ik}$. The capacity penalty is $B\sum_{i<k}x_{ij}x_{kj}$: it penalizes pairs of containers assigned to the same position.

The notebook adds three objective terms: a unit cost for moving a container, a negative reward for placing containers with the same destination in the same bay, and a simple weight-distribution preference. Together, these terms and the penalties define the energy $E(x)$. In the code, the upper-triangular matrix `Q` stores this energy using the convention

$$E(x)=\sum_{v=0}^{K-1}Q_{vv}x_v+\sum_{0\leq u<v<K}Q_{uv}x_ux_v,$$

where $K=NM$ and variable index $v=iM+j$ corresponds to container $i$ and position $j$. Thus, off-diagonal coefficients are counted once; they are not interpreted as both entries of a symmetric matrix. `QUBOFormulator.formulate_qubo` builds this matrix. The penalty weights are modeling parameters: they must be large enough relative to objective terms to discourage violations, while feasibility is also checked explicitly when samples are decoded.

### QAOA and the simulation
A QAOA state is produced by alternating a problem operation, which gives computational-basis states phases based on their QUBO energies, and a mixing operation, which changes their amplitudes. For depth $p$, the idealized state is

$$|\boldsymbol{\gamma},\boldsymbol{\beta}\rangle = U_M(\beta_p)U_P(\gamma_p)\cdots U_M(\beta_1)U_P(\gamma_1)|+\rangle^{\otimes K},$$

where $U_P(\gamma)=\exp(-i\gamma H_P)$, $H_P|x\rangle=E(x)|x\rangle$, and $U_M(\beta)=\exp(-i\beta\sum_{q=0}^{K-1}X_q)$. Here $|+\rangle=(|0\rangle+|1\rangle)/\sqrt{2}$, $X_q$ is the Pauli-X operator on qubit $q$, and the QUBO energy supplies the diagonal problem Hamiltonian. The `QAOAOptimizer` applies these operations, samples measurement outcomes, evaluates their energies, and searches over angles. Its parameter search is a simple stochastic perturbation method, not gradient descent.

The state-vector simulator stores all $2^K$ amplitudes explicitly. Applying the problem operation therefore evaluates the energy for every basis state, and measurement samples from the resulting probability distribution. The `noise_level` setting represents a simple mixture of the simulated measurement probabilities with a uniform distribution; it is not a physical noise-channel or hardware model. For this instance, the encoding requires one qubit per container-position variable, so the configuration must use 16 qubits.

In [ ]:
from dataclasses import dataclass
import math
import time
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Container:
    """Container data used by the re-stow assignment model."""

    id: str
    destination_port: str
    weight: float
    current_position: Tuple[int, int, int]
    discharge_sequence: int


@dataclass
class Position:
    """A candidate stow position and its descriptive attributes."""

    bay: int
    row: int
    tier: int
    accessible: bool = True
    max_weight: float = 50.0
    occupied: bool = False


@dataclass
class QuantumConfig:
    """Settings for the QAOA state-vector demonstration."""

    num_qubits: int
    qaoa_depth: int = 3
    optimization_iterations: int = 100
    learning_rate: float = 0.01
    measurement_shots: int = 1000
    noise_level: float = 0.01


@dataclass
class QuantumResult:
    """Best feasible sampled assignment and its simulation diagnostics."""

    best_solution: Optional[Dict[str, Position]]
    best_cost: float
    best_probability: float
    optimization_history: List[float]
    quantum_circuit_depth: int
    computation_time: float
    measurement_distribution: Dict[str, float]
    parameter_history: List[float]

In [ ]:
class QuantumSimulator:
    """State-vector simulator for the gates used in this QAOA example."""

    def __init__(self, num_qubits: int):
        if num_qubits < 1:
            raise ValueError("The simulator requires at least one qubit.")
        self.num_qubits = num_qubits
        self.state = None
        self.gate_history = []
        self.noise_level = 0.0

    def initialize_state(self, initial_state: str = "uniform"):
        """Prepare a uniform superposition or the all-zero basis state."""
        dimension = 2 ** self.num_qubits
        if initial_state == "uniform":
            self.state = np.ones(dimension, dtype=complex) / np.sqrt(dimension)
        elif initial_state == "zero":
            self.state = np.zeros(dimension, dtype=complex)
            self.state[0] = 1.0
        else:
            raise ValueError("initial_state must be 'uniform' or 'zero'.")
        self.gate_history = []
        self.noise_level = 0.0

    def apply_hadamard(self, qubit: int):
        """Apply a Hadamard gate using the simulator's qubit convention."""
        gate = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
        self._apply_single_qubit_gate(gate, qubit, "H")

    def apply_rotation_z(self, qubit: int, angle: float):
        """Apply the single-qubit Z rotation used in circuit demonstrations."""
        gate = np.diag([np.exp(-1j * angle / 2), np.exp(1j * angle / 2)])
        self._apply_single_qubit_gate(gate, qubit, f"Rz({angle:.3f})")

    def apply_rotation_x(self, qubit: int, angle: float):
        """Apply the mixer rotation to one qubit."""
        gate = np.array(
            [
                [np.cos(angle / 2), -1j * np.sin(angle / 2)],
                [-1j * np.sin(angle / 2), np.cos(angle / 2)],
            ],
            dtype=complex,
        )
        self._apply_single_qubit_gate(gate, qubit, f"Rx({angle:.3f})")

    def apply_cnot(self, control_qubit: int, target_qubit: int):
        """Apply a CNOT by permuting basis amplitudes."""
        self._check_qubit(control_qubit)
        self._check_qubit(target_qubit)
        if control_qubit == target_qubit:
            raise ValueError("Control and target qubits must be different.")
        control_bit = self.num_qubits - 1 - control_qubit
        target_bit = self.num_qubits - 1 - target_qubit
        indices = np.arange(2 ** self.num_qubits)
        flip = ((indices >> control_bit) & 1).astype(bool)
        sources = indices ^ (flip.astype(int) << target_bit)
        updated_state = np.empty_like(self.state)
        updated_state[indices] = self.state[sources]
        self.state = updated_state
        self.gate_history.append(("CNOT", control_qubit, target_qubit))

    def _check_qubit(self, qubit: int):
        if not 0 <= qubit < self.num_qubits:
            raise ValueError("Qubit index is outside the simulated register.")

    def _apply_single_qubit_gate(self, gate: np.ndarray, qubit: int, gate_name: str):
        """Apply a gate without constructing an exponentially large matrix."""
        if self.state is None:
            raise RuntimeError("Initialize the quantum state before applying gates.")
        self._check_qubit(qubit)
        tensor_state = self.state.reshape((2,) * self.num_qubits)
        moved_state = np.moveaxis(tensor_state, qubit, 0)
        transformed = gate @ moved_state.reshape(2, -1)
        restored = np.moveaxis(
            transformed.reshape(moved_state.shape), 0, qubit
        )
        self.state = restored.reshape(-1)
        self.gate_history.append((gate_name, qubit))

    def measure_state(self, shots: int = 1000) -> Dict[str, float]:
        """Sample bit strings and return their observed relative frequencies."""
        if self.state is None:
            raise RuntimeError("Initialize the quantum state before measurement.")
        if shots < 1:
            raise ValueError("shots must be a positive integer.")
        probabilities = np.abs(self.state) ** 2
        probabilities = (1 - self.noise_level) * probabilities
        probabilities += self.noise_level / len(probabilities)
        probabilities /= probabilities.sum()
        samples = np.random.choice(len(probabilities), size=shots, p=probabilities)
        values, counts = np.unique(samples, return_counts=True)
        return {
            format(int(value), f"0{self.num_qubits}b"): int(count) / shots
            for value, count in zip(values, counts)
        }

    def get_circuit_depth(self) -> int:
        """Return the number of recorded gate or problem-operation applications."""
        return len(self.gate_history)

    def add_noise(self, noise_level: float):
        """Set the uniform-mixture measurement-noise approximation."""
        if not 0 <= noise_level <= 1:
            raise ValueError("noise_level must lie between 0 and 1.")
        self.noise_level = noise_level

In [ ]:
class QUBOFormulator:
    """Build and decode the vessel re-stow QUBO assignment model."""

    def __init__(self, containers: List[Container], positions: List[Position]):
        if not containers or not positions:
            raise ValueError("At least one container and one position are required.")
        self.containers = containers
        self.positions = positions
        self.num_containers = len(containers)
        self.num_positions = len(positions)

    def formulate_qubo(self) -> Tuple[np.ndarray, Dict]:
        """Create an upper-triangular QUBO matrix and variable metadata."""
        num_binary_vars = self.num_containers * self.num_positions
        Q = np.zeros((num_binary_vars, num_binary_vars), dtype=float)
        penalty_A = 10.0
        penalty_B = 8.0

        # The squared exactly-one penalty yields diagonal -A and pair terms 2A.
        for i in range(self.num_containers):
            for j in range(self.num_positions):
                var = i * self.num_positions + j
                Q[var, var] -= penalty_A
                for other_j in range(j + 1, self.num_positions):
                    other_var = i * self.num_positions + other_j
                    Q[var, other_var] += 2 * penalty_A

        # A positive pair penalty prevents two containers using one position.
        for j in range(self.num_positions):
            for i in range(self.num_containers):
                var = i * self.num_positions + j
                for other_i in range(i + 1, self.num_containers):
                    other_var = other_i * self.num_positions + j
                    Q[var, other_var] += penalty_B

        move_cost = 1.0
        segregation_reward = -2.0
        weight_factor = 0.5
        for i, container in enumerate(self.containers):
            for j, position in enumerate(self.positions):
                var = i * self.num_positions + j
                coordinates = (position.bay, position.row, position.tier)
                if coordinates != container.current_position:
                    Q[var, var] += move_cost
                if container.weight > 25:
                    Q[var, var] += (
                        -weight_factor if position.bay in [2, 3, 4]
                        else weight_factor
                    )

        # Add each same-destination, same-bay reward once per assignment pair.
        for i in range(self.num_containers):
            for other_i in range(i + 1, self.num_containers):
                if (
                    self.containers[i].destination_port
                    != self.containers[other_i].destination_port
                ):
                    continue
                for j, position in enumerate(self.positions):
                    for other_j, other_position in enumerate(self.positions):
                        if position.bay == other_position.bay:
                            var = i * self.num_positions + j
                            other_var = other_i * self.num_positions + other_j
                            row, column = sorted((var, other_var))
                            Q[row, column] += segregation_reward

        aux_info = {
            "num_binary_vars": num_binary_vars,
            "num_containers": self.num_containers,
            "num_positions": self.num_positions,
            "constraint_weights": {
                "assignment": penalty_A,
                "capacity": penalty_B,
            },
            "objective_weights": {
                "move_cost": move_cost,
                "segregation": segregation_reward,
                "weight_factor": weight_factor,
            },
            "variable_mapping": self._create_variable_mapping(),
        }
        return Q, aux_info

    def _create_variable_mapping(self) -> Dict:
        """Map each binary-variable index to its container and position."""
        mapping = {}
        for i, container in enumerate(self.containers):
            for j, position in enumerate(self.positions):
                var = i * self.num_positions + j
                mapping[var] = {
                    "container": container.id,
                    "position": (position.bay, position.row, position.tier),
                }
        return mapping

    def extract_solution_from_binary(
        self, binary_string: str, aux_info: Dict
    ) -> Dict[str, Position]:
        """Decode selected bits into a container-to-position assignment."""
        if len(binary_string) != aux_info["num_binary_vars"]:
            raise ValueError("Bit-string length must equal the number of QUBO variables.")
        solution = {}
        variable_mapping = aux_info["variable_mapping"]
        for index, bit in enumerate(binary_string):
            if bit == "1":
                entry = variable_mapping[index]
                bay, row, tier = entry["position"]
                solution[entry["container"]] = Position(bay, row, tier)
        return solution

    def is_feasible(self, solution: Dict[str, Position]) -> bool:
        """Check exactly-one assignment and at-most-one position occupancy."""
        if set(solution) != {container.id for container in self.containers}:
            return False
        coordinates = [
            (position.bay, position.row, position.tier)
            for position in solution.values()
        ]
        available = {
            (position.bay, position.row, position.tier)
            for position in self.positions
        }
        return len(set(coordinates)) == len(coordinates) and set(coordinates) <= available

    def calculate_solution_cost(self, solution: Dict[str, Position]) -> float:
        """Return the fixed-dollar move cost for a feasible assignment."""
        if not self.is_feasible(solution):
            raise ValueError("Move cost is defined here only for feasible assignments.")
        moved = 0
        for container in self.containers:
            position = solution[container.id]
            coordinates = (position.bay, position.row, position.tier)
            moved += coordinates != container.current_position
        return 150.0 * moved

In [ ]:
class QAOAOptimizer:
    """Run a small QAOA state-vector simulation with stochastic angle search."""

    def __init__(self, config: QuantumConfig):
        self.config = config
        self.quantum_simulator = QuantumSimulator(config.num_qubits)
        self.optimization_history = []
        self.parameter_history = []

    def optimize(self, Q: np.ndarray, aux_info: Dict) -> QuantumResult:
        """Sample QAOA states and retain the best feasible measured assignment."""
        if Q.shape[0] != Q.shape[1]:
            raise ValueError("Q must be a square QUBO matrix.")
        if self.config.num_qubits != Q.shape[0]:
            raise ValueError("This encoding requires one qubit per QUBO variable.")
        if self.config.qaoa_depth < 1 or self.config.optimization_iterations < 1:
            raise ValueError("Depth and iteration count must be positive.")

        start_time = time.time()
        gamma_params = np.random.uniform(0, 2 * np.pi, self.config.qaoa_depth)
        beta_params = np.random.uniform(0, np.pi, self.config.qaoa_depth)
        energies = self._all_state_energies(Q)
        best_cost = float("inf")
        best_solution = None
        best_probability = 0.0
        best_expected_energy = float("inf")
        best_gamma = gamma_params.copy()
        best_beta = beta_params.copy()
        measurement_results = {}
        formulator = QUBOFormulator.__new__(QUBOFormulator)

        print(
            f"Starting QAOA simulation with {self.config.num_qubits} qubits "
            f"and depth {self.config.qaoa_depth}."
        )
        for iteration in range(self.config.optimization_iterations):
            self._apply_qaoa_circuit(Q, gamma_params, beta_params, energies)
            measurement_results = self.quantum_simulator.measure_state(
                self.config.measurement_shots
            )
            expected_energy = sum(
                probability * self._calculate_qubo_cost(state, Q)
                for state, probability in measurement_results.items()
            )
            if expected_energy < best_expected_energy:
                best_expected_energy = expected_energy
                best_gamma = gamma_params.copy()
                best_beta = beta_params.copy()

            current_cost, current_solution, current_probability = (
                self._evaluate_measurements(
                    measurement_results, aux_info, Q, formulator
                )
            )
            if current_solution is not None and (
                current_cost < best_cost
                or (
                    math.isclose(current_cost, best_cost, rel_tol=1e-9, abs_tol=1e-9)
                    and current_probability > best_probability
                )
            ):
                best_cost = current_cost
                best_solution = current_solution
                best_probability = current_probability

            self.optimization_history.append(best_cost)
            self.parameter_history.append(best_expected_energy)
            if (iteration + 1) % 20 == 0:
                print(
                    f"Iteration {iteration + 1:3d}: best feasible QUBO energy "
                    f"= {best_cost:.3f}; sampled expected energy "
                    f"= {expected_energy:.3f}"
                )
            gamma_params, beta_params = self._update_parameters(
                best_gamma, best_beta
            )

        return QuantumResult(
            best_solution=best_solution,
            best_cost=best_cost,
            best_probability=best_probability,
            optimization_history=self.optimization_history.copy(),
            quantum_circuit_depth=self.quantum_simulator.get_circuit_depth(),
            computation_time=time.time() - start_time,
            measurement_distribution=measurement_results,
            parameter_history=self.parameter_history.copy(),
        )

    def _all_state_energies(self, Q: np.ndarray) -> np.ndarray:
        """Evaluate the upper-triangular QUBO energy for every basis state."""
        dimension = 2 ** self.config.num_qubits
        indices = np.arange(dimension, dtype=np.uint64)
        bits = np.empty((dimension, self.config.num_qubits), dtype=float)
        for variable in range(self.config.num_qubits):
            shift = self.config.num_qubits - 1 - variable
            bits[:, variable] = (indices >> shift) & 1
        energies = bits @ np.diag(Q)
        for first in range(self.config.num_qubits):
            for second in range(first + 1, self.config.num_qubits):
                if Q[first, second] != 0:
                    energies += Q[first, second] * bits[:, first] * bits[:, second]
        return energies

    def _apply_qaoa_circuit(
        self,
        Q: np.ndarray,
        gamma_params: np.ndarray,
        beta_params: np.ndarray,
        energies: np.ndarray,
    ):
        """Prepare the initial state and apply alternating QAOA operations."""
        self.quantum_simulator.initialize_state("uniform")
        for layer in range(self.config.qaoa_depth):
            self._apply_problem_hamiltonian(energies, gamma_params[layer])
            self._apply_mixing_hamiltonian(beta_params[layer])
        self.quantum_simulator.add_noise(self.config.noise_level)

    def _apply_problem_hamiltonian(self, energies: np.ndarray, gamma: float):
        """Apply the diagonal phase exp(-i gamma E(x)) to each basis state."""
        self.quantum_simulator.state *= np.exp(-1j * gamma * energies)
        self.quantum_simulator.gate_history.append(("Problem phase", gamma))

    def _apply_mixing_hamiltonian(self, beta: float):
        """Apply the standard transverse-field X mixer to every qubit."""
        for qubit in range(self.config.num_qubits):
            self.quantum_simulator.apply_rotation_x(qubit, 2 * beta)

    def _evaluate_measurements(
        self,
        measurement_results: Dict[str, float],
        aux_info: Dict,
        Q: np.ndarray,
        formulator: QUBOFormulator,
) -> Tuple[float, Optional[Dict[str, Position]], float]:
        """Select the lowest-energy feasible assignment among sampled states."""
        best_cost = float("inf")
        best_solution = None
        best_probability = 0.0
        variable_mapping = aux_info["variable_mapping"]
        for binary_string, probability in measurement_results.items():
            solution = formulator.extract_solution_from_binary(binary_string, aux_info)
            if not formulator.is_feasible(solution):
                continue
            cost = self._calculate_qubo_cost(binary_string, Q)
            if cost < best_cost or (
                math.isclose(cost, best_cost, rel_tol=1e-9, abs_tol=1e-9)
                and probability > best_probability
            ):
                best_cost = cost
                best_solution = solution
                best_probability = probability
        return best_cost, best_solution, best_probability

    def _calculate_qubo_cost(self, binary_string: str, Q: np.ndarray) -> float:
        """Evaluate a bit string using the QUBO's upper-triangular convention."""
        if len(binary_string) != Q.shape[0]:
            raise ValueError("Bit-string length must equal the QUBO dimension.")
        bits = np.fromiter((int(bit) for bit in binary_string), dtype=float)
        cost = float(np.dot(np.diag(Q), bits))
        for first in range(len(bits)):
            for second in range(first + 1, len(bits)):
                cost += Q[first, second] * bits[first] * bits[second]
        return cost

    def _update_parameters(
        self, gamma_params: np.ndarray, beta_params: np.ndarray
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Perturb the best observed angles to explore nearby parameter values."""
        gamma_params = gamma_params + np.random.normal(
            0, self.config.learning_rate, len(gamma_params)
        )
        beta_params = beta_params + np.random.normal(
            0, self.config.learning_rate, len(beta_params)
        )
        gamma_params = np.mod(gamma_params, 2 * np.pi)
        beta_params = np.mod(beta_params, np.pi)
        return gamma_params, beta_params

### Create the demonstration instance
The instance has four containers and four candidate positions. Since each container-position assignment has its own binary variable, this formulation requires $4\times4=16$ qubits. The simulator configuration is set from the QUBO variable count rather than using a shorter register that could not encode every assignment.

In [ ]:
def create_quantum_problem():
    """Create the small container and position instance used in the example."""
    containers = [
        Container("HKG-1", "HKG", 20.0, (1, 1, 1), 1),
        Container("SHA-1", "SHA", 18.0, (1, 1, 2), 3),
        Container("HKG-2", "HKG", 22.0, (2, 1, 1), 2),
        Container("SHA-2", "SHA", 19.0, (2, 1, 2), 4),
    ]
    positions = [
        Position(bay, 1, tier)
        for bay in [1, 2]
        for tier in [1, 2]
    ]
    return containers, positions


containers, positions = create_quantum_problem()
qubo_formulator = QUBOFormulator(containers, positions)
num_binary_vars = len(containers) * len(positions)
config = QuantumConfig(
    num_qubits=num_binary_vars,
    qaoa_depth=3,
    optimization_iterations=100,
    learning_rate=0.01,
    measurement_shots=1000,
    noise_level=0.01,
)

assert config.num_qubits == num_binary_vars
print("QAOA demonstration configuration")
print(f"  Containers: {len(containers)}")
print(f"  Positions: {len(positions)}")
print(f"  Binary variables and qubits: {config.num_qubits}")
print(f"  QAOA depth: {config.qaoa_depth}")
print(f"  Parameter-search iterations: {config.optimization_iterations}")
print(f"  Measurement shots per iteration: {config.measurement_shots}")

In [ ]:
print("\n=== QUBO FORMULATION ===")
Q, aux_info = qubo_formulator.formulate_qubo()
assert Q.shape == (config.num_qubits, config.num_qubits)
print(f"QUBO matrix shape: {Q.shape}")
print(f"Binary variables: {aux_info['num_binary_vars']}")
print(f"Constraint weights: {aux_info['constraint_weights']}")
print(f"Objective weights: {aux_info['objective_weights']}")
print("\nQUBO matrix (first 8 rows and columns):")
print(Q[:8, :8].round(2))

nonzero_count = np.count_nonzero(Q)
matrix_size = Q.shape[0] * Q.shape[1]
sparsity = 1 - nonzero_count / matrix_size
print("\nQUBO matrix characteristics:")
print(f"  Nonzero coefficients: {nonzero_count}")
print(f"  Sparsity: {sparsity:.2%}")
print(f"  Density: {1 - sparsity:.2%}")

plt.figure(figsize=(8, 6))
image = plt.imshow(Q, cmap="RdBu_r", aspect="auto")
plt.colorbar(image, label="QUBO coefficient")
plt.title("Upper-Triangular QUBO Coefficients")
plt.xlabel("Binary variable index")
plt.ylabel("Binary variable index")
plt.tight_layout()
plt.show()

In [ ]:
print("\n=== QAOA OPTIMIZATION ===")
qaoa_optimizer = QAOAOptimizer(config)
quantum_result = qaoa_optimizer.optimize(Q, aux_info)

print("\n=== SIMULATION RESULTS ===")
print(f"Computation time: {quantum_result.computation_time:.2f} seconds")
print(f"Recorded circuit operations: {quantum_result.quantum_circuit_depth}")
print(f"Parameter-search iterations: {len(quantum_result.optimization_history)}")
if quantum_result.best_solution is None:
    print("No feasible assignment was observed in the sampled bit strings.")
else:
    print(f"Best feasible QUBO energy: {quantum_result.best_cost:.3f}")
    print(f"Observed frequency of that bit string: {quantum_result.best_probability:.3f}")
    for container in containers:
        position = quantum_result.best_solution[container.id]
        coordinates = (position.bay, position.row, position.tier)
        moved = coordinates != container.current_position
        move_status = "Move" if moved else "No move"
        print(
            f"  {container.id}: bay {position.bay}, row {position.row}, "
            f"tier {position.tier} ({move_status})"
        )
    actual_move_cost = qubo_formulator.calculate_solution_cost(
        quantum_result.best_solution
    )
    print(f"Fixed-charge move cost: ${actual_move_cost:.2f}")

assert len(quantum_result.optimization_history) == config.optimization_iterations
if quantum_result.best_solution is not None:
    assert qubo_formulator.is_feasible(quantum_result.best_solution)

In [ ]:
def visualize_quantum_results(result: QuantumResult):
    """Plot sampled optimization diagnostics without adding comparison data."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    axes[0].plot(result.optimization_history, color="navy", linewidth=1.8)
    axes[0].set_title("Best Feasible QUBO Energy")
    axes[0].set_xlabel("Parameter-search iteration")
    axes[0].set_ylabel("QUBO energy")
    axes[0].grid(True, alpha=0.3)

    top_measurements = sorted(
        result.measurement_distribution.items(),
        key=lambda item: item[1],
        reverse=True,
    )[:10]
    if top_measurements:
        states, frequencies = zip(*top_measurements)
        axes[1].bar(range(len(states)), frequencies, color="purple", alpha=0.75)
        axes[1].set_xticks(range(len(states)))
        axes[1].set_xticklabels(states, rotation=45, ha="right")
    axes[1].set_title("Most Frequent Final Measurements")
    axes[1].set_xlabel("Measured bit string")
    axes[1].set_ylabel("Observed frequency")

    axes[2].plot(result.parameter_history, color="darkgreen", linewidth=1.8)
    axes[2].set_title("Sampled Expected QUBO Energy")
    axes[2].set_xlabel("Parameter-search iteration")
    axes[2].set_ylabel("Estimated energy")
    axes[2].grid(True, alpha=0.3)

    fig.suptitle("QAOA State-Vector Simulation Diagnostics")
    fig.tight_layout()
    plt.show()


visualize_quantum_results(quantum_result)

The first plot tracks the lowest-energy feasible bit string observed up to each iteration. It can remain unchanged when no better feasible sample appears; that does not mean the QAOA state or its expected energy is unchanged. The third plot records the estimated expected QUBO energy from the measurement samples at each iteration and is therefore subject to sampling variation. The final measurement plot reports frequencies from the last circuit evaluation, not probabilities guaranteed by the underlying ideal quantum state.

In [ ]:
print("\n=== QUBO AND MOVE-COST INTERPRETATION ===")
if quantum_result.best_solution is None:
    print(
        "No feasible bit string was observed. Increase sampling or adjust the "
        "QAOA settings before interpreting assignment quality."
    )
else:
    move_cost = qubo_formulator.calculate_solution_cost(
        quantum_result.best_solution
    )
    print(f"Feasible sampled QUBO energy: {quantum_result.best_cost:.3f}")
    print(f"Fixed-charge move cost: ${move_cost:.2f}")
    print(
        "These values measure different quantities: QUBO energy includes "
        "constraint penalties and weighted preferences, while move cost "
        "counts only containers assigned away from their current positions."
    )
print(
    "No classical baseline measurements are included in this demonstration; "
    "therefore it does not support a runtime or solution-quality comparison."
)

In [ ]:
def analyze_quantum_scalability():
    """Show how this assignment encoding grows with containers and positions."""
    scenarios = [
        {"name": "Small illustrative instance", "containers": 4, "positions": 4},
        {"name": "Example scale A", "containers": 20, "positions": 20},
        {"name": "Example scale B", "containers": 50, "positions": 50},
        {"name": "Example scale C", "containers": 100, "positions": 100},
    ]
    print("\n=== ENCODING AND STATE-VECTOR SCALING ===")
    print(
        f"Current instance: {len(containers)} containers, {len(positions)} positions, "
        f"{aux_info['num_binary_vars']} binary variables, {config.num_qubits} qubits."
    )
    print(
        f"{'Scenario':<28} {'Containers':>11} {'Positions':>10} "
        f"{'Variables / qubits':>20} {'State amplitudes':>20}"
    )
    print("-" * 94)
    for scenario in scenarios:
        variables = scenario["containers"] * scenario["positions"]
        amplitudes = 2 ** variables
        print(
            f"{scenario['name']:<28} {scenario['containers']:>11} "
            f"{scenario['positions']:>10} {variables:>20} {amplitudes:>20,}"
        )
    print(
        "The state-vector simulator stores 2^K amplitudes for K binary variables. "
        "This exponential storage requirement limits the sizes that can be "
        "simulated directly; the table is an encoding calculation, not a "
        "hardware forecast or a quantum-advantage estimate."
    )
    return scenarios


scalability_analysis = analyze_quantum_scalability()

In [ ]:
def generate_quantum_insights(result: QuantumResult, formulator: QUBOFormulator):
    """Summarize what the sampled result does and does not establish."""
    print("\n=== INTERPRETING THE QAOA DEMONSTRATION ===")
    if result.best_solution is None:
        print("No feasible assignment appeared in the sampled measurement outcomes.")
    else:
        assert formulator.is_feasible(result.best_solution)
        print(f"Best observed feasible QUBO energy: {result.best_cost:.3f}")
        print(
            "The reported assignment satisfies the modeled exactly-one and "
            "at-most-one position constraints. Its measured frequency is an "
            "empirical frequency from finite shots."
        )
    print("\nInterpretation and limitations:")
    print("  - QAOA combines a parameterized quantum-state simulation with classical angle updates.")
    print("  - This implementation uses stochastic perturbations, not gradient-based optimization.")
    print("  - The state-vector calculation scales exponentially in the number of qubits.")
    print("  - The simplified noise setting is a measurement-probability mixture, not a hardware model.")
    print("  - No speedup, approximation guarantee, or comparison with another solver is measured here.")


generate_quantum_insights(quantum_result, qubo_formulator)

## Results and interpretation
The notebook builds a QUBO assignment model, encodes one binary variable per container-position pair, simulates QAOA on a state vector, and evaluates measured bit strings against the model's feasibility conditions. Outputs are produced when the notebook is run; no particular optimization result is assumed here.

A feasible sampled assignment is a candidate, not by itself a proof of global optimality. The reported QUBO energy includes penalties and assignment preferences, whereas the fixed-charge move cost counts only relocations. They should not be compared as if they shared units or represented the same objective. A run that finds no feasible sample has not produced a usable assignment.

### Scope of conclusions
This example illustrates the mechanics of a QUBO-based QAOA workflow and the exponential state-vector cost of directly simulating it. It does not establish that QAOA is faster or better than a classical method, that its sampled result is optimal, or that the simulated noise reflects a quantum device. Such claims would require a defined experimental protocol and measured, reproducible comparisons that are outside this notebook's scope.

### Modeling extensions
The assignment model can be expanded to represent operational constraints such as accessibility, stacking compatibility, weight limits, and discharge order. Each additional requirement would need a precise mathematical definition and a corresponding QUBO contribution or feasibility rule. Larger instances also require careful consideration of the encoding and the computational cost of the chosen simulation or execution environment.